# 02 - Data Cleaning & Splits

Quality control, taxonomy validation via WoRMS, and stratified train/test splits.

**Input**: `data/raw/merged_barcodes.csv`
**Output**: `data/processed/` with pre_training.csv, supervised_train.csv, supervised_val.csv, supervised_test.csv, unseen.csv

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kluless/marinemamba/blob/main/notebooks/02_data_cleaning.ipynb)

In [ ]:
# === COLAB SETUP ===
# !pip install requests pandas scikit-learn tqdm
# !git clone https://github.com/kluless/marinemamba.git
# %cd marinemamba

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import re
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import time

RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Load merged data
df = pd.read_csv(RAW_DIR / "merged_barcodes.csv")
print(f"Loaded {len(df)} records")

## 1. Sequence Quality Filters

In [ ]:
initial_count = len(df)

# Uppercase sequences
df["nucleotides"] = df["nucleotides"].str.upper().str.strip()

# Remove non-DNA characters (keep only A, C, G, T, N)
df["nucleotides"] = df["nucleotides"].str.replace(r"[^ACGTN]", "", regex=True)

# Filter: minimum length 500bp
df["seq_len"] = df["nucleotides"].str.len()
df = df[df["seq_len"] >= 500].copy()
print(f"After min length (500bp): {len(df)} ({initial_count - len(df)} removed)")

# Filter: maximum length 700bp (COI barcodes should be ~658bp)
df = df[df["seq_len"] <= 700].copy()
print(f"After max length (700bp): {len(df)}")

# Filter: max 5% ambiguous bases (N)
df["n_frac"] = df["nucleotides"].str.count("N") / df["seq_len"]
df = df[df["n_frac"] <= 0.05].copy()
print(f"After N filter (<=5%): {len(df)}")

# Remove records without species name
df = df.dropna(subset=["species_name"])
df = df[df["species_name"].str.strip() != ""].copy()
print(f"After removing unnamed: {len(df)}")

# Remove species with only genus-level ID (single word)
df = df[df["species_name"].str.contains(" ", na=False)].copy()
print(f"After requiring binomial names: {len(df)}")

print(f"\nTotal removed: {initial_count - len(df)} ({100*(initial_count-len(df))/initial_count:.1f}%)")

## 2. Taxonomy Validation via WoRMS

In [ ]:
def validate_worms(species_name, cache=None):
    """Validate species name against WoRMS. Returns accepted name or None."""
    if cache and species_name in cache:
        return cache[species_name]
    
    try:
        url = "https://www.marinespecies.org/rest/AphiaRecordsByName/" + species_name.replace(" ", "%20")
        r = requests.get(url, params={"like": "false", "marine_only": "true"}, timeout=10)
        if r.status_code == 200 and r.text.strip():
            records = r.json()
            if records:
                accepted = next(
                    (rec for rec in records if rec.get("status") == "accepted"),
                    records[0]
                )
                result = {
                    "valid_name": accepted.get("valid_name", species_name),
                    "aphia_id": accepted.get("AphiaID"),
                    "family": accepted.get("family"),
                    "order": accepted.get("order"),
                    "class": accepted.get("class")
                }
                if cache is not None:
                    cache[species_name] = result
                return result
    except Exception:
        pass
    
    if cache is not None:
        cache[species_name] = None
    return None

# Validate unique species (not every record — save API calls)
unique_species = df["species_name"].unique()
print(f"Validating {len(unique_species)} unique species against WoRMS...")

worms_cache = {}
validated = 0
for sp in tqdm(unique_species):
    result = validate_worms(sp, cache=worms_cache)
    if result:
        validated += 1
    time.sleep(0.2)  # WoRMS rate limiting

print(f"Validated: {validated}/{len(unique_species)} ({100*validated/len(unique_species):.1f}%)")

# Save cache for reuse
with open(PROC_DIR / "worms_cache.json", "w") as f:
    json.dump({k: v for k, v in worms_cache.items() if v is not None}, f, indent=2)

In [ ]:
# Apply validated taxonomy
def apply_worms(row):
    result = worms_cache.get(row["species_name"])
    if result:
        return pd.Series({
            "species_name": result["valid_name"],
            "genus_name": result["valid_name"].split()[0],
            "family_name": result.get("family", row.get("family_name", "")),
            "order_name": result.get("order", row.get("order_name", ""))
        })
    return pd.Series({
        "species_name": row["species_name"],
        "genus_name": row.get("genus_name", row["species_name"].split()[0]),
        "family_name": row.get("family_name", ""),
        "order_name": row.get("order_name", "")
    })

taxonomy_cols = df.apply(apply_worms, axis=1)
df[["species_name", "genus_name", "family_name", "order_name"]] = taxonomy_cols
print(f"Taxonomy updated. {df['species_name'].nunique()} unique species.")

## 3. Pad/Truncate to 660bp

In [ ]:
MAX_LEN = 660

def pad_or_truncate(seq, max_len=MAX_LEN):
    """Left-pad with N if shorter, truncate if longer. Matches BarcodeMamba format."""
    if len(seq) >= max_len:
        return seq[:max_len]
    return "N" * (max_len - len(seq)) + seq

df["nucleotides"] = df["nucleotides"].apply(pad_or_truncate)
print(f"All sequences padded/truncated to {MAX_LEN}bp")
print(f"Length check: min={df['nucleotides'].str.len().min()}, max={df['nucleotides'].str.len().max()}")

## 4. Create Splits

In [ ]:
# Identify genera to hold out for zero-shot evaluation
# Criteria: genera with 5+ species AND not in the top-50 most common genera
genus_species_count = df.groupby("genus_name")["species_name"].nunique()
genus_total_seqs = df.groupby("genus_name").size()

# Candidate holdout genera: 5+ species, not too dominant
candidates = genus_species_count[(genus_species_count >= 5) & (genus_species_count <= 50)]
top50 = genus_total_seqs.nlargest(50).index
candidates = candidates[~candidates.index.isin(top50)]

# Randomly select ~10% of candidate genera for holdout
np.random.seed(42)
n_holdout = max(50, len(candidates) // 10)
holdout_genera = np.random.choice(candidates.index, size=min(n_holdout, len(candidates)), replace=False)

print(f"Holding out {len(holdout_genera)} genera for zero-shot evaluation")
print(f"Holdout genera examples: {list(holdout_genera[:10])}")

In [ ]:
# Split data
unseen_mask = df["genus_name"].isin(holdout_genera)
unseen = df[unseen_mask].copy()
seen = df[~unseen_mask].copy()

print(f"Seen: {len(seen)} records, {seen['species_name'].nunique()} species")
print(f"Unseen: {len(unseen)} records, {unseen['species_name'].nunique()} species, {unseen['genus_name'].nunique()} genera")

# Pre-training set: ALL seen data (used for NTP, labels not needed)
pretrain = seen[["nucleotides", "processid"]].copy()

# Supervised splits: stratified by species
# Filter to species with >= 3 samples (need enough for train/val/test)
species_counts = seen["species_name"].value_counts()
valid_species = species_counts[species_counts >= 3].index
supervised = seen[seen["species_name"].isin(valid_species)].copy()

print(f"\nSupervised set: {len(supervised)} records, {supervised['species_name'].nunique()} species (>= 3 samples each)")

# 70/10/20 split
train_val, test = train_test_split(
    supervised, test_size=0.2, random_state=42, stratify=supervised["species_name"]
)
train, val = train_test_split(
    train_val, test_size=0.125, random_state=42, stratify=train_val["species_name"]
)  # 0.125 of 0.8 = 0.1 of total

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)} | Unseen: {len(unseen)}")

In [ ]:
# Save all splits (BarcodeMamba CSV format)
output_cols = ["nucleotides", "species_name", "genus_name", "family_name", "order_name", "processid"]

pretrain.to_csv(PROC_DIR / "pre_training.csv", index=False)
train[output_cols].to_csv(PROC_DIR / "supervised_train.csv", index=False)
val[output_cols].to_csv(PROC_DIR / "supervised_val.csv", index=False)
test[output_cols].to_csv(PROC_DIR / "supervised_test.csv", index=False)
unseen[output_cols].to_csv(PROC_DIR / "unseen.csv", index=False)

# Save stats
stats = {
    "total_sequences": len(df),
    "total_species": int(df["species_name"].nunique()),
    "total_genera": int(df["genus_name"].nunique()),
    "total_families": int(df["family_name"].nunique()),
    "pretrain_size": len(pretrain),
    "train_size": len(train),
    "val_size": len(val),
    "test_size": len(test),
    "unseen_size": len(unseen),
    "unseen_genera": int(unseen["genus_name"].nunique()),
    "n_classes": int(supervised["species_name"].nunique()),
    "holdout_genera": list(holdout_genera)
}

with open(PROC_DIR / "dataset_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("\nAll splits saved to data/processed/")
print(json.dumps(stats, indent=2))

## Next

Proceed to `03_baselines.ipynb` to establish BLAST and ML baselines.